In [1]:
!pip install scipy -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ISNN_outputs/'

Mounted at /content/drive


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

In [4]:
OUT = '/content/ISNN_outputs/'
os.makedirs(OUT, exist_ok=True)
print(f'Output directory: {OUT}')

Output directory: /content/ISNN_outputs/


In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


## Data Generation:

In [6]:
def additive_fn(X):
    """
    Eq. (12):  f = exp(-0.5x) + log(1+exp(0.4y)) + tanh(t) + sin(z) - 0.4
    Convex in x,  convex+mono in y,  mono in t,  arbitrary in z.
    Input order: X[:,0]=x, X[:,1]=y, X[:,2]=t, X[:,3]=z
    """
    return (np.exp(-0.5 * X[:, 0])
            + np.log1p(np.exp(0.4 * X[:, 1]))
            + np.tanh(X[:, 2])
            + np.sin(X[:, 3])
            - 0.4).reshape(-1, 1)

def multiplicative_fn(X):
    """
    Eq. (13)-(14):  g = f_x * f_y * f_t * f_z
    f_x = exp(-0.3x),  f_y = (0.15y)^2,
    f_t = tanh(0.3t),  f_z = 0.2*sin(0.5z+2)+0.5
    """
    fx = np.exp(-0.3 * X[:, 0])
    fy = (0.15 * X[:, 1]) ** 2
    ft = np.tanh(0.3 * X[:, 2])
    fz = 0.2 * np.sin(0.5 * X[:, 3] + 2) + 0.5
    return (fx * fy * ft * fz).reshape(-1, 1)

def lhs_sample(n, d, lo, hi, seed):
    sampler = qmc.LatinHypercube(d=d, seed=seed)
    s = sampler.random(n)
    return lo + s * (hi - lo)


def make_datasets():
    """
    Toy problem 1 – Additive:
      Train: 500 pts, domain [0,4]^4   (LHS)
      Test : 5000 pts, domain [0,6]^4  (LHS)

    Toy problem 2 – Multiplicative:
      Train: 500 pts, domain [0,4]^4   (LHS)
      Test : 5000 pts, domain [0,10]^4 (LHS)
    """
    Xtr_a = lhs_sample(500,  4, 0, 4,  42)
    Xte_a = lhs_sample(5000, 4, 0, 6,  43)
    ytr_a = additive_fn(Xtr_a)
    yte_a = additive_fn(Xte_a)

    Xtr_m = lhs_sample(500,  4, 0, 4,  44)
    Xte_m = lhs_sample(5000, 4, 0, 10, 45)
    ytr_m = multiplicative_fn(Xtr_m)
    yte_m = multiplicative_fn(Xte_m)

    return (Xtr_a, ytr_a, Xte_a, yte_a,
            Xtr_m, ytr_m, Xte_m, yte_m)



## Pytorch Models

In [7]:
def softplus(x):  return torch.log1p(torch.exp(x))
def sigmoid(x):   return torch.sigmoid(x)


def clamp_non_negative_(module):
    with torch.no_grad():
        module.weight.clamp_(min=0.0)


class ISNN1_PT(nn.Module):
    def __init__(self, n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2):
        super().__init__()
        self.Hx, self.Hy, self.Ht, self.Hz = Hx, Hy, Ht, Hz

        self.Wy  = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(Hy)])
        self.Wz  = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(Hz)])
        self.Wt  = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(Ht)])

        self.Wx0 = nn.Linear(1,        n_hidden)
        self.Wxy = nn.Linear(n_hidden, n_hidden, bias=False)
        self.Wxz = nn.Linear(n_hidden, n_hidden, bias=False)
        self.Wxt = nn.Linear(n_hidden, n_hidden, bias=False)

        self.Wxx = nn.ModuleList([nn.Linear(n_hidden, n_hidden) for _ in range(Hx - 1)])
        self.out  = nn.Linear(n_hidden, 1)

    def forward(self, X):
        x0 = X[:, 0:1]; y0 = X[:, 1:2]
        t0 = X[:, 2:3]; z0 = X[:, 3:4]
        yh = y0
        for layer in self.Wy:
            yh = softplus(layer(yh))
        zh = z0
        for layer in self.Wz:
            zh = sigmoid(layer(zh))
        th = t0
        for layer in self.Wt:
            th = sigmoid(layer(th))
        F0 = self.Wx0(x0) + self.Wxy(yh) + self.Wxz(zh) + self.Wxt(th)
        xh = softplus(F0)
        for layer in self.Wxx:
            xh = softplus(layer(xh))
        return self.out(xh)

    def enforce_constraints(self):
        for layer in self.Wxx: clamp_non_negative_(layer)
        for layer in self.Wy:  clamp_non_negative_(layer)
        clamp_non_negative_(self.Wxy)
        for layer in self.Wt:  clamp_non_negative_(layer)
        clamp_non_negative_(self.Wxt)

In [8]:
class ISNN2_PT(nn.Module):
    def __init__(self, n_hidden=15, H=2):
        super().__init__()
        self.H = H

        self.Wy = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(H - 1)])
        self.Wz = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(H - 1)])
        self.Wt = nn.ModuleList([nn.Linear(n_hidden if h > 0 else 1, n_hidden) for h in range(H - 1)])

        self.Wxx0_0 = nn.Linear(n_hidden, n_hidden, bias=False)
        self.Wxx0_0 = nn.Linear(1,        n_hidden, bias=True)
        self.Wxy_0  = nn.Linear(1,        n_hidden, bias=False)
        self.Wxz_0  = nn.Linear(1,        n_hidden, bias=False)
        self.Wxt_0  = nn.Linear(1,        n_hidden, bias=False)

        self.Wxx  = nn.ModuleList([nn.Linear(n_hidden, n_hidden, bias=False) for _ in range(H - 1)])
        self.Wxx0 = nn.ModuleList([nn.Linear(1,        n_hidden, bias=False) for _ in range(H - 1)])
        self.Wxy  = nn.ModuleList([nn.Linear(n_hidden, n_hidden, bias=False) for _ in range(H - 1)])
        self.Wxz  = nn.ModuleList([nn.Linear(n_hidden, n_hidden, bias=False) for _ in range(H - 1)])
        self.Wxt  = nn.ModuleList([nn.Linear(n_hidden, n_hidden, bias=False) for _ in range(H - 1)])
        self.bx   = nn.ParameterList([nn.Parameter(torch.zeros(n_hidden))   for _ in range(H - 1)])

        self.out  = nn.Linear(n_hidden, 1)

    def forward(self, X):
        x0 = X[:, 0:1]; y0 = X[:, 1:2]
        t0 = X[:, 2:3]; z0 = X[:, 3:4]
        yh = y0
        for layer in self.Wy:
            yh = softplus(layer(yh))
        zh = z0
        for layer in self.Wz:
            zh = sigmoid(layer(zh))
        th = t0
        for layer in self.Wt:
            th = sigmoid(layer(th))
        F0 = (self.Wxx0_0(x0) + self.Wxy_0(y0) + self.Wxz_0(z0) + self.Wxt_0(t0))
        xh = softplus(F0)
        for i in range(self.H - 1):
            Fh = (self.Wxx[i](xh) + self.Wxx0[i](x0) + self.Wxy[i](yh) + self.Wxz[i](zh) + self.Wxt[i](th) + self.bx[i])
            xh = softplus(Fh)
        return self.out(xh)

    def enforce_constraints(self):
        for l in self.Wy:  clamp_non_negative_(l)
        for l in self.Wt:  clamp_non_negative_(l)
        for l in self.Wxx: clamp_non_negative_(l)
        clamp_non_negative_(self.Wxy_0)
        for l in self.Wxy: clamp_non_negative_(l)
        clamp_non_negative_(self.Wxt_0)
        for l in self.Wxt: clamp_non_negative_(l)

In [9]:
class FFNN_PT(nn.Module):
    def __init__(self, n_in=4, n_hidden=30, n_layers=2):
        super().__init__()
        layers = [nn.Linear(n_in, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers.append(nn.Linear(n_hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, X):
        return self.net(X)

In [10]:
def train_pytorch(model, Xtr, ytr, Xte, yte, epochs=30000, lr=1e-3, log_every=100):
    model = model.to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr)
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).to(DEVICE)
    Xte_t = torch.tensor(Xte, dtype=torch.float32).to(DEVICE)
    yte_t = torch.tensor(yte, dtype=torch.float32).to(DEVICE)

    train_loss_hist, test_loss_hist, epoch_hist = [], [], []

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        pred = model(Xtr_t)
        loss = nn.functional.mse_loss(pred, ytr_t)
        loss.backward()
        opt.step()
        if hasattr(model, 'enforce_constraints'):
            model.enforce_constraints()
        train_loss_hist.append(loss.item())
        if ep % log_every == 0 or ep == 1:
            model.eval()
            with torch.no_grad():
                tl = nn.functional.mse_loss(model(Xte_t), yte_t).item()
            test_loss_hist.append(tl)
            epoch_hist.append(ep)

    return np.array(train_loss_hist), np.array(test_loss_hist), np.array(epoch_hist)

## Numpy Models

In [11]:
def np_softplus(x):
    return np.log1p(np.exp(np.clip(x, -500, 500)))

def np_softplus_d(x):
    return 1.0 / (1.0 + np.exp(np.clip(-x, -500, 500)))

def np_sigmoid(x):
    return 1.0 / (1.0 + np.exp(np.clip(-x, -500, 500)))

def np_sigmoid_d(x):
    s = np_sigmoid(x)
    return s * (1.0 - s)

In [12]:
class AdamParam:
    def __init__(self, val, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, clamp_min=None):
        self.val = val.copy()
        self.m   = np.zeros_like(val)
        self.v   = np.zeros_like(val)
        self.lr  = lr; self.b1 = b1; self.b2 = b2; self.eps = eps
        self.t   = 0
        self.clamp_min = clamp_min

    def step(self, grad):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * grad
        self.v = self.b2 * self.v + (1 - self.b2) * grad**2
        mh = self.m / (1 - self.b1**self.t)
        vh = self.v / (1 - self.b2**self.t)
        self.val -= self.lr * mh / (np.sqrt(vh) + self.eps)
        if self.clamp_min is not None:
            self.val = np.maximum(self.val, self.clamp_min)


def xavier(in_d, out_d):
    lim = np.sqrt(6.0 / (in_d + out_d))
    return np.random.uniform(-lim, lim, (out_d, in_d))

def zeros_b(n): return np.zeros(n)

In [13]:
class ISNN1_NP:
    def __init__(self, n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2, lr=1e-3, seed=0):
        np.random.seed(seed)
        nh  = n_hidden
        pos = 0.0

        self.Wy  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr) for h in range(Hy)]
        self.by  = [AdamParam(zeros_b(nh), lr) for _ in range(Hy)]
        self.Wz  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr) for h in range(Hz)]
        self.bz  = [AdamParam(zeros_b(nh), lr) for _ in range(Hz)]
        self.Wt  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr, clamp_min=pos) for h in range(Ht)]
        self.bt  = [AdamParam(zeros_b(nh), lr) for _ in range(Ht)]

        self.Wx0  = AdamParam(xavier(1,  nh), lr)
        self.Wxy  = AdamParam(xavier(nh, nh), lr, clamp_min=pos)
        self.Wxz  = AdamParam(xavier(nh, nh), lr)
        self.Wxt  = AdamParam(xavier(nh, nh), lr, clamp_min=pos)
        self.bx0  = AdamParam(zeros_b(nh),    lr)

        self.Wxx  = [AdamParam(xavier(nh, nh), lr, clamp_min=pos) for _ in range(Hx - 1)]
        self.bxx  = [AdamParam(zeros_b(nh),    lr) for _ in range(Hx - 1)]

        self.Wout = AdamParam(xavier(nh, 1), lr)
        self.bout = AdamParam(zeros_b(1),    lr)

    def forward(self, X):
        x0 = X[:, 0:1]; y0 = X[:, 1:2]
        t0 = X[:, 2:3]; z0 = X[:, 3:4]
        self._yZ, self._yA = [], [y0]
        ah = y0
        for Wy, by in zip(self.Wy, self.by):
            z = ah @ Wy.val.T + by.val
            self._yZ.append(z); ah = np_softplus(z); self._yA.append(ah)
        yHy = ah
        self._zZ, self._zA = [], [z0]
        ah = z0
        for Wz, bz in zip(self.Wz, self.bz):
            z = ah @ Wz.val.T + bz.val
            self._zZ.append(z); ah = np_sigmoid(z); self._zA.append(ah)
        zHz = ah
        self._tZ, self._tA = [], [t0]
        ah = t0
        for Wt, bt in zip(self.Wt, self.bt):
            z = ah @ Wt.val.T + bt.val
            self._tZ.append(z); ah = np_sigmoid(z); self._tA.append(ah)
        tHt = ah
        F0 = (x0 @ self.Wx0.val.T + yHy @ self.Wxy.val.T + zHz @ self.Wxz.val.T + tHt @ self.Wxt.val.T + self.bx0.val)
        self._xF = [F0]
        ah = np_softplus(F0)
        self._xA = [x0, ah]

        for Wxx, bxx in zip(self.Wxx, self.bxx):
            z = ah @ Wxx.val.T + bxx.val
            self._xF.append(z); ah = np_softplus(z); self._xA.append(ah)
        out = ah @ self.Wout.val.T + self.bout.val
        self._xA.append(out)
        return out

    def backward_and_update(self, X, y_true):
        B   = X.shape[0]
        out = self._xA[-1]
        loss = np.mean((out - y_true)**2)
        dL   = 2.0 * (out - y_true) / B
        a_last = self._xA[-2]
        gWout  = dL.T @ a_last
        gbout  = dL.sum(axis=0)
        dL_xa  = dL @ self.Wout.val
        self.Wout.step(gWout); self.bout.step(gbout)
        delta = dL_xa
        for h in reversed(range(len(self.Wxx))):
            dact = delta * np_softplus_d(self._xF[h + 1])
            gW   = dact.T @ self._xA[h + 1]
            gb   = dact.sum(axis=0)
            delta = dact @ self.Wxx[h].val
            self.Wxx[h].step(gW); self.bxx[h].step(gb)

        x0 = X[:, 0:1]; yHy = self._yA[-1]
        zHz = self._zA[-1]; tHt = self._tA[-1]
        dact_F0 = delta * np_softplus_d(self._xF[0])
        gWx0  = dact_F0.T @ x0
        gWxy  = dact_F0.T @ yHy
        gWxz  = dact_F0.T @ zHz
        gWxt  = dact_F0.T @ tHt
        gbx0  = dact_F0.sum(axis=0)
        self.Wx0.step(gWx0); self.Wxy.step(gWxy)
        self.Wxz.step(gWxz); self.Wxt.step(gWxt)
        self.bx0.step(gbx0)

        dL_yHy = dact_F0 @ self.Wxy.val
        dL_zHz = dact_F0 @ self.Wxz.val
        dL_tHt = dact_F0 @ self.Wxt.val

        delta_y = dL_yHy
        for h in reversed(range(len(self.Wy))):
            dact = delta_y * np_softplus_d(self._yZ[h])
            gW   = dact.T @ self._yA[h]; gb = dact.sum(axis=0)
            delta_y = dact @ self.Wy[h].val
            self.Wy[h].step(gW); self.by[h].step(gb)

        delta_z = dL_zHz
        for h in reversed(range(len(self.Wz))):
            dact = delta_z * np_sigmoid_d(self._zZ[h])
            gW   = dact.T @ self._zA[h]; gb = dact.sum(axis=0)
            delta_z = dact @ self.Wz[h].val
            self.Wz[h].step(gW); self.bz[h].step(gb)

        delta_t = dL_tHt
        for h in reversed(range(len(self.Wt))):
            dact = delta_t * np_sigmoid_d(self._tZ[h])
            gW   = dact.T @ self._tA[h]; gb = dact.sum(axis=0)
            delta_t = dact @ self.Wt[h].val
            self.Wt[h].step(gW); self.bt[h].step(gb)

        return loss

    def predict(self, X): return self.forward(X)
    def train_step(self, X, y): self.forward(X); return self.backward_and_update(X, y)


In [14]:
class ISNN2_NP:
    def __init__(self, n_hidden=15, H=2, lr=1e-3, seed=0):
        np.random.seed(seed)
        nh  = n_hidden
        pos = 0.0
        self.H = H

        self.Wy  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr, clamp_min=pos) for h in range(H - 1)]
        self.by  = [AdamParam(zeros_b(nh), lr) for _ in range(H - 1)]
        self.Wz  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr) for h in range(H - 1)]
        self.bz  = [AdamParam(zeros_b(nh), lr) for _ in range(H - 1)]
        self.Wt  = [AdamParam(xavier(1 if h == 0 else nh, nh), lr, clamp_min=pos) for h in range(H - 1)]
        self.bt  = [AdamParam(zeros_b(nh), lr) for _ in range(H - 1)]

        self.Wxx0_0 = AdamParam(xavier(1,  nh), lr)
        self.Wxy_0  = AdamParam(xavier(1,  nh), lr, clamp_min=pos)
        self.Wxz_0  = AdamParam(xavier(1,  nh), lr)
        self.Wxt_0  = AdamParam(xavier(1,  nh), lr, clamp_min=pos)
        self.bx_0   = AdamParam(zeros_b(nh), lr)

        self.Wxx  = [AdamParam(xavier(nh, nh), lr, clamp_min=pos) for _ in range(H - 1)]
        self.Wxx0 = [AdamParam(xavier(1,  nh), lr)               for _ in range(H - 1)]
        self.Wxy  = [AdamParam(xavier(nh, nh), lr, clamp_min=pos) for _ in range(H - 1)]
        self.Wxz  = [AdamParam(xavier(nh, nh), lr)               for _ in range(H - 1)]
        self.Wxt  = [AdamParam(xavier(nh, nh), lr, clamp_min=pos) for _ in range(H - 1)]
        self.bx   = [AdamParam(zeros_b(nh), lr)                  for _ in range(H - 1)]

        self.Wout = AdamParam(xavier(nh, 1), lr)
        self.bout = AdamParam(zeros_b(1),    lr)

    def forward(self, X):
        x0 = X[:, 0:1]; y0 = X[:, 1:2]
        t0 = X[:, 2:3]; z0 = X[:, 3:4]
        self._yZ, self._yA = [], [y0]
        ah = y0
        for Wy, by in zip(self.Wy, self.by):
            z = ah @ Wy.val.T + by.val
            self._yZ.append(z); ah = np_softplus(z); self._yA.append(ah)
        yh_final = ah
        self._zZ, self._zA = [], [z0]
        ah = z0
        for Wz, bz in zip(self.Wz, self.bz):
            z = ah @ Wz.val.T + bz.val
            self._zZ.append(z); ah = np_sigmoid(z); self._zA.append(ah)
        zh_final = ah
        self._tZ, self._tA = [], [t0]
        ah = t0
        for Wt, bt in zip(self.Wt, self.bt):
            z = ah @ Wt.val.T + bt.val
            self._tZ.append(z); ah = np_sigmoid(z); self._tA.append(ah)
        th_final = ah
        F0 = (x0  @ self.Wxx0_0.val.T + y0 @ self.Wxy_0.val.T + z0 @ self.Wxz_0.val.T + t0 @ self.Wxt_0.val.T + self.bx_0.val)
        xh = np_softplus(F0)
        self._xF = [F0]
        self._xA = [xh]

        for i in range(self.H - 1):
            Fh = (xh @ self.Wxx[i].val.T + x0 @ self.Wxx0[i].val.T + yh_final @ self.Wxy[i].val.T + zh_final @ self.Wxz[i].val.T + th_final @ self.Wxt[i].val.T + self.bx[i].val)
            xh = np_softplus(Fh)
            self._xF.append(Fh); self._xA.append(xh)

        out = xh @ self.Wout.val.T + self.bout.val
        self._x_last = xh
        return out

    def backward_and_update(self, X, y_true):
        B    = X.shape[0]
        x0   = X[:, 0:1]; y0 = X[:, 1:2]
        t0   = X[:, 2:3]; z0 = X[:, 3:4]
        out  = self._x_last @ self.Wout.val.T + self.bout.val
        loss = np.mean((out - y_true)**2)
        dL   = 2.0 * (out - y_true) / B

        gWout = dL.T @ self._x_last
        gbout = dL.sum(0)
        delta = dL @ self.Wout.val
        self.Wout.step(gWout); self.bout.step(gbout)

        yh_f = self._yA[-1]; zh_f = self._zA[-1]; th_f = self._tA[-1]
        dL_yh = np.zeros_like(yh_f)
        dL_zh = np.zeros_like(zh_f)
        dL_th = np.zeros_like(th_f)

        for i in reversed(range(self.H - 1)):
            dact  = delta * np_softplus_d(self._xF[i + 1])
            gWxx  = dact.T @ self._xA[i]
            gWxx0 = dact.T @ x0
            gWxy  = dact.T @ yh_f
            gWxz  = dact.T @ zh_f
            gWxt  = dact.T @ th_f
            gb    = dact.sum(0)
            delta  = dact @ self.Wxx[i].val
            dL_yh += dact @ self.Wxy[i].val
            dL_zh += dact @ self.Wxz[i].val
            dL_th += dact @ self.Wxt[i].val
            self.Wxx[i].step(gWxx); self.Wxx0[i].step(gWxx0)
            self.Wxy[i].step(gWxy); self.Wxz[i].step(gWxz)
            self.Wxt[i].step(gWxt); self.bx[i].step(gb)

        dact_F0  = delta * np_softplus_d(self._xF[0])
        gWxx0_0  = dact_F0.T @ x0
        gWxy_0   = dact_F0.T @ y0
        gWxz_0   = dact_F0.T @ z0
        gWxt_0   = dact_F0.T @ t0
        gbx_0    = dact_F0.sum(0)
        dL_yh   += dact_F0 @ self.Wxy_0.val
        dL_zh   += dact_F0 @ self.Wxz_0.val
        dL_th   += dact_F0 @ self.Wxt_0.val
        self.Wxx0_0.step(gWxx0_0); self.Wxy_0.step(gWxy_0)
        self.Wxz_0.step(gWxz_0);   self.Wxt_0.step(gWxt_0)
        self.bx_0.step(gbx_0)

        delta_y = dL_yh
        for h in reversed(range(len(self.Wy))):
            dact = delta_y * np_softplus_d(self._yZ[h])
            gW   = dact.T @ self._yA[h]; gb = dact.sum(0)
            delta_y = dact @ self.Wy[h].val
            self.Wy[h].step(gW); self.by[h].step(gb)

        delta_z = dL_zh
        for h in reversed(range(len(self.Wz))):
            dact = delta_z * np_sigmoid_d(self._zZ[h])
            gW   = dact.T @ self._zA[h]; gb = dact.sum(0)
            delta_z = dact @ self.Wz[h].val
            self.Wz[h].step(gW); self.bz[h].step(gb)

        delta_t = dL_th
        for h in reversed(range(len(self.Wt))):
            dact = delta_t * np_sigmoid_d(self._tZ[h])
            gW   = dact.T @ self._tA[h]; gb = dact.sum(0)
            delta_t = dact @ self.Wt[h].val
            self.Wt[h].step(gW); self.bt[h].step(gb)

        return loss

    def predict(self, X): return self.forward(X)
    def train_step(self, X, y): self.forward(X); return self.backward_and_update(X, y)

In [15]:
class FFNN_NP:
    def __init__(self, n_in=4, n_hidden=30, n_layers=2, lr=1e-3, seed=0):
        np.random.seed(seed)
        dims = [n_in] + [n_hidden] * n_layers + [1]
        self.W = [AdamParam(xavier(dims[i], dims[i + 1]), lr) for i in range(len(dims) - 1)]
        self.b = [AdamParam(zeros_b(dims[i + 1]), lr)         for i in range(len(dims) - 1)]

    def forward(self, X):
        self._Z, self._A = [], [X]
        ah = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = ah @ W.val.T + b.val
            self._Z.append(z)
            ah = np.tanh(z) if i < len(self.W) - 1 else z
            self._A.append(ah)
        return ah

    def backward_and_update(self, X, y_true):
        B    = X.shape[0]
        out  = self._A[-1]
        loss = np.mean((out - y_true)**2)
        dL   = 2.0 * (out - y_true) / B
        gW = dL.T @ self._A[-2]; gb = dL.sum(0)
        delta = dL @ self.W[-1].val
        self.W[-1].step(gW); self.b[-1].step(gb)

        for i in reversed(range(len(self.W) - 1)):
            dact  = delta * (1 - np.tanh(self._Z[i])**2)
            gW    = dact.T @ self._A[i]; gb = dact.sum(0)
            delta = dact @ self.W[i].val
            self.W[i].step(gW); self.b[i].step(gb)
        return loss

    def predict(self, X): return self.forward(X)
    def train_step(self, X, y): self.forward(X); return self.backward_and_update(X, y)


## Training loops

In [16]:
def train_numpy(model, Xtr, ytr, Xte, yte, epochs=30000, log_every=100):
    train_hist, test_hist, epoch_hist = [], [], []
    for ep in range(1, epochs + 1):
        loss = model.train_step(Xtr, ytr)
        train_hist.append(loss)
        if ep % log_every == 0 or ep == 1:
            yp = model.predict(Xte)
            tl = np.mean((yp - yte)**2)
            test_hist.append(tl)
            epoch_hist.append(ep)
    return np.array(train_hist), np.array(test_hist), np.array(epoch_hist)



In [17]:
N_RUNS = 10
EPOCHS = 30000
LR     = 1e-3


def run_all_seeds_pt(ModelClass, kwargs, Xtr, ytr, Xte, yte):
    all_tr, all_te = [], []
    for s in range(N_RUNS):
        torch.manual_seed(s)
        m = ModelClass(**kwargs)
        tr, te, ep = train_pytorch(m, Xtr, ytr, Xte, yte, epochs=EPOCHS, lr=LR)
        all_tr.append(tr); all_te.append(te)
    return np.array(all_tr), np.array(all_te), ep


def run_all_seeds_np(ModelClass, kwargs, Xtr, ytr, Xte, yte):
    all_tr, all_te = [], []
    for s in range(N_RUNS):
        m = ModelClass(**kwargs, seed=s)
        tr, te, ep = train_numpy(m, Xtr, ytr, Xte, yte, epochs=EPOCHS)
        all_tr.append(tr); all_te.append(te)
    return np.array(all_tr), np.array(all_te), ep

In [18]:
COLOR = {'FFNN': 'red', 'ISNN-1': 'blue', 'ISNN-2': 'green'}
LABEL = {'FFNN': 'FFNN', 'ISNN-1': 'ISNN-1', 'ISNN-2': 'ISNN-2'}


def plot_mean_std(ax, data, x_vals, name, log_y=True):
    mu  = data.mean(axis=0)
    std = data.std(axis=0)
    c   = COLOR[name]
    ax.plot(x_vals, mu, color=c, label=LABEL[name], linewidth=1.5)
    ax.fill_between(x_vals, mu - std, mu + std, color=c, alpha=0.25)
    if log_y: ax.set_yscale('log')


def make_loss_figure(results_tr, results_te, ep_tr, ep_te, title_prefix, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for name in ['FFNN', 'ISNN-1', 'ISNN-2']:
        plot_mean_std(axes[0], results_tr[name], ep_tr, name)
        plot_mean_std(axes[1], results_te[name], ep_te, name)

    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Training Loss (MSE)', fontsize=12)
    axes[0].set_title(f'(a) {title_prefix} – Training Loss', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11); axes[0].grid(True, which='both', alpha=0.3)

    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Test Loss (MSE)', fontsize=12)
    axes[1].set_title(f'(b) {title_prefix} – Test Loss', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=11); axes[1].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'saved {filename}')


def make_behavior_figure(pt_models, fn_true, lo, hi, train_bound,
                         ylabel, title_suffix, filename):
    t_all  = np.linspace(lo, hi, 300)
    X_diag = np.column_stack([t_all] * 4).astype(np.float32)
    y_true = fn_true(X_diag)

    n = len(pt_models)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]

    for ax, (name, model) in zip(axes, pt_models):
        model.eval()
        X_diag_dev = torch.tensor(X_diag).to(DEVICE)
        with torch.no_grad():
            ypred_all = model(X_diag_dev).cpu().numpy()

        mask_in  = t_all <= train_bound
        mask_out = t_all >  train_bound

        ax.plot(t_all, y_true, 'k--', linewidth=2, label='True response')
        ax.plot(t_all[mask_in],  ypred_all[mask_in],  color='blue',  linewidth=2, label='Interpolated response')
        y_extrap = ypred_all[mask_out]
        ax.plot(t_all[mask_out], y_extrap, color='red', linewidth=2, label='Extrapolated response')
        ax.fill_between(t_all[mask_out],
                        y_extrap.flatten() - 0.15 * np.abs(y_extrap.flatten()).clip(0.05),
                        y_extrap.flatten() + 0.15 * np.abs(y_extrap.flatten()).clip(0.05),
                        color='red', alpha=0.30)

        ax.set_xlabel('x = y = t = z', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'({name}) {title_suffix}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'saved {filename}')

In [19]:
def make_numpy_behavior_figure(np_models, fn_true, lo, hi, train_bound,
                                ylabel, title_suffix, filename):
    t_all  = np.linspace(lo, hi, 300)
    X_diag = np.column_stack([t_all] * 4)
    y_true = fn_true(X_diag)

    n = len(np_models)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1: axes = [axes]

    for ax, (name, model) in zip(axes, np_models):
        ypred_all = model.predict(X_diag)
        mask_in   = t_all <= train_bound
        mask_out  = t_all >  train_bound

        ax.plot(t_all, y_true, 'k--', linewidth=2, label='True response')
        ax.plot(t_all[mask_in],  ypred_all[mask_in].flatten(),  color='blue',  linewidth=2, label='Interpolated response')
        y_extrap = ypred_all[mask_out].flatten()
        ax.plot(t_all[mask_out], y_extrap, color='red', linewidth=2, label='Extrapolated response')
        ax.fill_between(t_all[mask_out],
                        y_extrap.flatten() - 0.15 * np.abs(y_extrap.flatten()).clip(0.05),
                        y_extrap.flatten() + 0.15 * np.abs(y_extrap.flatten()).clip(0.05),
                        color='red', alpha=0.30)

        ax.set_xlabel('x = y = t = z', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'({name}) [NumPy] {title_suffix}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9, loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'saved {filename}')

In [ ]:
def main():
    print('=' * 65)
    print('  ISNN IMPLEMENTATION  –  Jadoon et al. (2025)')
    print('=' * 65)

    # datasets
    print('\n[1] Generating datasets (LHS sampling)...')
    (Xtr_a, ytr_a, Xte_a, yte_a,
     Xtr_m, ytr_m, Xte_m, yte_m) = make_datasets()
    print(f'    Additive  – train {Xtr_a.shape}  test {Xte_a.shape}')
    print(f'    Multiplic – train {Xtr_m.shape}  test {Xte_m.shape}')

    for name, arr in [('Xtr_additive', Xtr_a), ('ytr_additive', ytr_a),
                       ('Xte_additive', Xte_a), ('yte_additive', yte_a),
                       ('Xtr_multiplic', Xtr_m), ('ytr_multiplic', ytr_m),
                       ('Xte_multiplic', Xte_m), ('yte_multiplic', yte_m)]:
        np.save(OUT + name + '.npy', arr)
    print('    Datasets saved.')

    ep_tr = np.arange(1, EPOCHS + 1)
    ep_te = np.arange(1, EPOCHS // 100 + 1) * 100

    #  PYTORCH  –  ADDITIVE
    print('\n[2] PyTorch training  –  Additive function')
    pt_res_tr_a, pt_res_te_a = {}, {}
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_PT,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_PT,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_PT,  dict(n_hidden=15, H=2)),
    ]:
        print(f'    {name} ...', end=' ', flush=True)
        tr_all, te_all, _ = run_all_seeds_pt(ModelCls, kw, Xtr_a, ytr_a, Xte_a, yte_a)
        pt_res_tr_a[name] = tr_all; pt_res_te_a[name] = te_all
        print(f'final test loss = {te_all[:, -1].mean():.4e}')

    make_loss_figure(pt_res_tr_a, pt_res_te_a, ep_tr, ep_te,
                     'Additive (PyTorch)', OUT + 'fig3_pytorch_additive_loss.png')

    pt_behav_a = []
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_PT,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_PT,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_PT,  dict(n_hidden=15, H=2)),
    ]:
        torch.manual_seed(0); m = ModelCls(**kw)
        train_pytorch(m, Xtr_a, ytr_a, Xte_a, yte_a, epochs=EPOCHS, lr=LR)
        pt_behav_a.append((name, m))

    make_behavior_figure(pt_behav_a, additive_fn, lo=0, hi=6, train_bound=4,
                         ylabel='f(x,y,t,z)', title_suffix='Additive (PyTorch)',
                         filename=OUT + 'fig4_pytorch_additive_behavior.png')


    #  PYTORCH  –  MULTIPLICATIVE
    print('\n[3] PyTorch training  –  Multiplicative function')
    pt_res_tr_m, pt_res_te_m = {}, {}
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_PT,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_PT,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_PT,  dict(n_hidden=15, H=2)),
    ]:
        print(f'    {name} ...', end=' ', flush=True)
        tr_all, te_all, _ = run_all_seeds_pt(ModelCls, kw, Xtr_m, ytr_m, Xte_m, yte_m)
        pt_res_tr_m[name] = tr_all; pt_res_te_m[name] = te_all
        print(f'final test loss = {te_all[:, -1].mean():.4e}')

    make_loss_figure(pt_res_tr_m, pt_res_te_m, ep_tr, ep_te,
                     'Multiplicative (PyTorch)', OUT + 'fig5_pytorch_multiplicative_loss.png')

    pt_behav_m = []
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_PT,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_PT,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_PT,  dict(n_hidden=15, H=2)),
    ]:
        torch.manual_seed(0); m = ModelCls(**kw)
        train_pytorch(m, Xtr_m, ytr_m, Xte_m, yte_m, epochs=EPOCHS, lr=LR)
        pt_behav_m.append((name, m))

    make_behavior_figure(pt_behav_m, multiplicative_fn, lo=0, hi=10, train_bound=4,
                         ylabel='g(x,y,t,z)', title_suffix='Multiplicative (PyTorch)',
                         filename=OUT + 'fig6_pytorch_multiplicative_behavior.png')


    #  NUMPY  –  ADDITIVE
    print('\n[4] NumPy (manual backprop) training  –  Additive')
    np_res_tr_a, np_res_te_a = {}, {}
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_NP,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_NP,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_NP,  dict(n_hidden=15, H=2)),
    ]:
        print(f'    {name} ...', end=' ', flush=True)
        tr_all, te_all, _ = run_all_seeds_np(ModelCls, kw, Xtr_a, ytr_a, Xte_a, yte_a)
        np_res_tr_a[name] = tr_all; np_res_te_a[name] = te_all
        print(f'final test loss = {te_all[:, -1].mean():.4e}')

    make_loss_figure(np_res_tr_a, np_res_te_a, ep_tr, ep_te,
                     'Additive (NumPy manual backprop)', OUT + 'fig3_numpy_additive_loss.png')

    np_behav_a = []
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_NP,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_NP,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_NP,  dict(n_hidden=15, H=2)),
    ]:
        m = ModelCls(**kw, seed=0)
        train_numpy(m, Xtr_a, ytr_a, Xte_a, yte_a, epochs=EPOCHS)
        np_behav_a.append((name, m))

    make_numpy_behavior_figure(np_behav_a, additive_fn, lo=0, hi=6, train_bound=4,
                                ylabel='f(x,y,t,z)', title_suffix='Additive',
                                filename=OUT + 'fig4_numpy_additive_behavior.png')

    #  NUMPY  –  MULTIPLICATIVE
    print('\n[5] NumPy (manual backprop) training  –  Multiplicative')
    np_res_tr_m, np_res_te_m = {}, {}
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_NP,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_NP,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_NP,  dict(n_hidden=15, H=2)),
    ]:
        print(f'    {name} ...', end=' ', flush=True)
        tr_all, te_all, _ = run_all_seeds_np(ModelCls, kw, Xtr_m, ytr_m, Xte_m, yte_m)
        np_res_tr_m[name] = tr_all; np_res_te_m[name] = te_all
        print(f'final test loss = {te_all[:, -1].mean():.4e}')

    make_loss_figure(np_res_tr_m, np_res_te_m, ep_tr, ep_te,
                     'Multiplicative (NumPy manual backprop)', OUT + 'fig5_numpy_multiplicative_loss.png')

    np_behav_m = []
    for name, ModelCls, kw in [
        ('FFNN',   FFNN_NP,   dict(n_in=4, n_hidden=30, n_layers=2)),
        ('ISNN-1', ISNN1_NP,  dict(n_hidden=10, Hx=2, Hy=2, Ht=2, Hz=2)),
        ('ISNN-2', ISNN2_NP,  dict(n_hidden=15, H=2)),
    ]:
        m = ModelCls(**kw, seed=0)
        train_numpy(m, Xtr_m, ytr_m, Xte_m, yte_m, epochs=EPOCHS)
        np_behav_m.append((name, m))

    make_numpy_behavior_figure(np_behav_m, multiplicative_fn, lo=0, hi=10, train_bound=4,
                                ylabel='g(x,y,t,z)', title_suffix='Multiplicative',
                                filename=OUT + 'fig6_numpy_multiplicative_behavior.png')

    print('\n' + '=' * 65)
    print(f'  All done!  Outputs saved to {OUT}')
    print('=' * 65)


if __name__ == '__main__':
    main()


  ISNN IMPLEMENTATION  –  Jadoon et al. (2025)

[1] Generating datasets (LHS sampling)...
    Additive  – train (500, 4)  test (5000, 4)
    Multiplic – train (500, 4)  test (5000, 4)
    Datasets saved.

[2] PyTorch training  –  Additive function
    FFNN ... final test loss = 1.5773e-01
    ISNN-1 ... 